In [ ]:
from enum import Enum
import cvxpy as cp
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.ticker import MaxNLocator
import pandas as pd

class Season(Enum):
	WINTER = 'jan'
	SPRING = 'apr'
	SUMMER = 'jul'
	AUTUMN = 'oct'

DAYS = 5					# number of days to model
I = 1						# number of containers to model
season = Season.AUTUMN		# season for RTM prices

# !!! OPERATIONAL PARAMETERS - DO NOT CHANGE !!!
DELTA_t = 1.0		# hr
HOURS_PER_DAY = 24
N = DAYS * HOURS_PER_DAY
SOC_MIN = 0.05		# dimensionless
SOC_MAX = 0.95		# dimensionless
Q_NEW = 3.916		# MWh, new container capacity
P_MAX = 0.979		# MW, hard limit
T_MIN = -30			# °C
T_REF = 25			# °C
T_MAX = 50			# °C
ETA = 0.968			# dimensionless

# FLEXIBLE MODEL PARAMS - CAN CHANGE FOR SENSITIVITY ANALYSIS
ALPHA = 2.61e-5		# MWh capacity lost per MWh throughput
BETA = 2.5			# °C / MWh throughput
GAMMA = 3.0			# °C / hr
KAPPA = 0.04		# / °C
A = 0.02			# dimensionless
B = 0.02			# dimensionless

# Day starting values for containers, spaced equally in range
SOH_0 = 0.9	# dimensionless
SOC_0 = 0.7	# dimensionless

# Day ending values
SOC_N = SOC_0

expected_prices_df = pd.read_csv("./data/raw/DAM/seasonal_stats.csv")
LAMBDA_DAILY = expected_prices_df[f'{season.value}_mean'].to_numpy().reshape(-1, 1)
LAMBDA = np.tile(LAMBDA_DAILY, (DAYS, 1))

def solve_bess_scenario(rho, sigma, aging):

	# states: k = 0 -> N, i = 1 -> 2
	E = cp.Variable((N + 1, I), nonneg=True)
	Q = cp.Variable((N + 1, I), nonneg=True)
	T = cp.Variable((N + 1, I), nonneg=True)

	# decisions: k = 0 -> N-1, i = 1 -> 2
	c = cp.Variable((N, I), nonneg=True)
	d = cp.Variable((N, I), nonneg=True)
	u = cp.Variable((N, I), nonneg=True)

	phi = cp.Parameter((N,I), nonneg=True)
	delta_Q = ALPHA * cp.multiply(phi, c + d) * DELTA_t

	constraints = [
		# Initial conditions
		Q[0, :] == Q_NEW * SOH_0,
		E[0, :] == cp.multiply(Q[0, :], SOC_0),
		T[0, :] == T_REF,
		# Dynamics - see below for capacity dynamics
		Q[1:, :] == Q[:-1, :] - delta_Q,
		E[1:, :] == E[:-1, :] + (ETA * c * DELTA_t) - (d * DELTA_t / ETA),
		T[1:, :] == T[:-1, :] + (BETA * (c + d) * DELTA_t) - (GAMMA * u * DELTA_t),
		# Boundary conditions
		# Q >= 0, # variable created as non-neg=True
		E >= Q * SOC_MIN,
		E <= Q * SOC_MAX,
		T >= T_MIN,
		T <= T_MAX,
		# c >= 0, # variable created as non-neg=True
		c <= P_MAX,
		# d >= 0, # variable created as non-neg=True
		d <= P_MAX,
		# u >= 0, # variable created as non-neg=True
		u <= 1,
		# Terminal conditions
		E[N, :] >= cp.multiply(Q[N, :], SOC_N)
	]

	arbitrage_cost = cp.sum(cp.multiply(LAMBDA, c - d) * DELTA_t)
	operating_cost = cp.sum(cp.multiply(LAMBDA, A + B * u) * P_MAX * DELTA_t)
	opportunity_cost = cp.sum(rho * delta_Q)
	replacement_cost = cp.sum(sigma * delta_Q)
	if (aging):
		objective = cp.Minimize(arbitrage_cost + operating_cost + opportunity_cost + replacement_cost)
	else:
		objective = cp.Minimize(arbitrage_cost + operating_cost)
	problem = cp.Problem(objective, constraints)

	T_est = np.full((N, I), T_REF)

	for iteration in range(10):
		print(f"Iteration {iteration} starting...")

		phi.value = 1 + KAPPA * np.maximum(0, T_est - T_REF)

		problem.solve(
			solver=cp.ECOS,
			feastol=1e-8
		)

		if problem.status == "optimal":
			T_est = T.value[:-1, :]
			print(f"Iteration {iteration}: Total Cost = {problem.value:.2f}")
		else:
			print("Solver failed to find an optimal solution.")
			break

	return Q.value

case1 = solve_bess_scenario(25000, 300000, True)
case2 = solve_bess_scenario(19500, 241000, True)
case3 = solve_bess_scenario(14000, 180000, True)
case4 = solve_bess_scenario(0, 0, True)


In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(case1.flatten() - case1[0,0], label=r"High Cost ($\rho$=\$25,000 / MWh, $\sigma$=\$300,000 / MWh)", color='red', linewidth=6)
plt.plot(case2.flatten() - case2[0,0], label=r"Baseline ($\rho$=\$19,500 / MWh, $\sigma$=\$241,000 / MWh)", color='lightgreen', linewidth=4)
plt.plot(case3.flatten() - case3[0,0], label=r"Low Cost ($\rho$=\$14,000 / MWh, $\sigma$=\$180,000 / MWh)", color='blue')
plt.plot(case4.flatten() - case4[0,0], label=r"Zero Cost ($\rho$=\$0 / MWh, $\sigma$=\$0 / MWh)", color='black', linestyle='--')

# Formatting the plot
plt.suptitle(r"Sensitivity Analysis: Opportunity & Replacement Pricing of Capacity Loss ($\Delta Q$)", fontsize=14)
plt.title("Autumn: Starting SoH 90%, SoC 70%", fontsize=11)
plt.xlabel("Time Step (Hour)")
plt.ylabel(r"Capacity Lost ($MWh$)")
plt.grid(True, alpha=0.3)
plt.legend()

# Display or save
plt.tight_layout()
plt.show()